# **IMPORTING DATASET**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("davidjfisher/illinois-doc-labeled-faces-dataset")

print("Path to dataset files:", path)

# **IMPORTING MODULES**

In [ ]:
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.express as px

from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

From the dataset create models to

1. Predict the Body Mass Index (BMI) using the facial images (frontal and side view) with 80-20 split.

Use 80% for training and 20% for testing.

2. Test the model with your face and your friend’s face to estimate BMI value.

3. Estimate the gender based on the facial features

4. Grade them based on the BMI as Underweight, Normal and Overweight (Obese)

5. The Metrics to be used are: MAE, MSE, R2, and Pearson Coefficient

6. Plot the distribution of offences that the inmates have done.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr


# **IMPORTING WARNINGS**

In [ ]:
import warnings
warnings.filterwarnings('always')
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="urllib3.contrib.pyopenssl")
warnings.filterwarnings("ignore", category=ResourceWarning)

# **READING PERSON.CSV**

In [ ]:
df = pd.read_csv('/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/person.csv',sep=';')
print(df.dtypes)

dropcols = ['name','projected_discharge_date', 'parole_date','electronic_detention_date','discharge_date','projected_parole_date','last_paroled_date','parent_institution','offender_status','location','sex_offender_registry_required','alias','Unnamed: 21']
df = df.drop(columns=dropcols)
pd.set_option('expand_frame_repr', False)
print('\n',df.head())

In [ ]:
df = df.dropna()
df

# **DATA CLEANING**

In [ ]:
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
df['admission_date'] = pd.to_datetime(df['admission_date'])

df['weight'] = pd.to_numeric(df['weight'])
df['height'] = pd.to_numeric(df['height'])

z_scores_height = (df['height'] - df['height'].mean()) / df['height'].std()
df = df.loc[abs(z_scores_height) < 3]

z_scores_weight = (df['weight'] - df['weight'].mean()) / df['weight'].std()
df = df.loc[abs(z_scores_weight) < 3]

print(f"Cleaned shape: {df.shape}")


# **Feature Engineering**


In [ ]:
df['height'] = df['height'] * 2.54 / 100
df['weight'] = df['weight'] * 0.453592
df['bmi'] = df['weight']/(df['height'])**2

def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 24.9:
        return 'Normal'
    else:
        return 'Overweight'

df['category'] = df['bmi'].apply(categorize_bmi)

# Convert timedelta to years
df['age'] = (df['admission_date'] - df['date_of_birth']).dt.total_seconds() / (365.25 * 24 * 3600)

# Ensure non-negative ages
df = df[df['age'] >= 0]

# Drop original date columns
dropcols = ['date_of_birth', 'admission_date']
df = df.drop(columns=dropcols)

# Remove outliers using z-score method
z_scores_age = (df['age'] - df['age'].mean()) / df['age'].std()
df = df.loc[abs(z_scores_age) < 3]

print(df.dtypes)

# **Data Visualization**

In [ ]:
df_ = pd.read_csv("/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/sentencing.csv", sep=';')

offense_counts = df_['offense'].value_counts()

top_offenses = offense_counts.head(10)

plt.figure(figsize=(10,6))
top_offenses.plot(kind='bar', color='lightcoral')
plt.title('Top 10 Offenses Distribution')
plt.xlabel('Offense')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Create a jointplot
joint = sns.jointplot(
    data=df,
    x='height',
    y='weight',
    hue='sex',
    kind='scatter',
    alpha=0.8,
    height=8
)

# Add titles and labels
joint.fig.suptitle('Scatterplot of Height vs Weight by Gender', fontsize=16)
joint.set_axis_labels('Height (m)', 'Weight (kg)', fontsize=12)
joint.ax_joint.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
male_subset = df[df['sex'] == 'Male']

plt.figure(figsize=(8, 5))
sns.kdeplot(data=male_subset, x='height', y='weight', cmap='Blues', fill=True)

plt.title('Density Plot of Height vs Weight for Males', fontsize=16)
plt.xlabel('Height (meters)', fontsize=12)
plt.ylabel('Weight (kilograms)', fontsize=12)
plt.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
female_subset = df[df['sex'] == 'Female']

plt.figure(figsize=(8, 5))
sns.histplot(data=female_subset, x='height', y='weight', bins=20, cbar=True, cmap='Purples')

plt.title('Height vs Weight Heatmap (Females Only)', fontsize=16)
plt.xlabel('Height (in meters)', fontsize=12)
plt.ylabel('Weight (in kilograms)', fontsize=12)
plt.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
np.random.seed(42)

males = df[df['sex'] == 'Male']
females = df[df['sex'] == 'Female']
sample_size = len(females)
print(sample_size)

equal_males = males.sample(sample_size)
balanced_dataset = pd.concat([equal_males, females])

plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=balanced_dataset,
    x='height',
    y='weight',
    hue='sex',
    fill=True,
    alpha=0.7,
    cmap='coolwarm'
)

plt.title('Height vs Weight Density (Balanced Dataset)', fontsize=16)
plt.xlabel('Height (meters)', fontsize=12)
plt.ylabel('Weight (kilograms)', fontsize=12)
plt.legend(title='Gender')
plt.grid(visible=True, alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='height', y='weight', hue='sex', palette='cool', bins=30, cbar=True)

plt.title('Distribution of Height and Weight by Gender', fontsize=16)
plt.xlabel('Height (in inches)', fontsize=12)
plt.ylabel('Weight (in pounds)', fontsize=12)
plt.grid(visible=True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(x='race', y='height', data=df, palette='mako', inner='quartile')
plt.title('Height Distribution Across Ethnic Groups', fontsize=16)
plt.ylabel('Height (in Inches)', fontsize=12)
plt.xlabel('Ethnic Group', fontsize=12)
plt.grid(visible=True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
def visualize_distribution(column, ax):
    df[column][df[column].notna()].value_counts().plot(
        kind='barh', ax=ax, color='teal'
    )
    ax.set_title(f"Distribution of {column.capitalize()}", fontsize=16)
    ax.set_ylabel(column.capitalize(), fontsize=12)
    ax.set_xlabel("Frequency", fontsize=12)
    return ax

features = ['race', 'hair', 'eyes']

num_features = len(features)
fig, axes = plt.subplots(1, num_features, figsize=(6 * num_features, 6))
fig.tight_layout(pad=3)

if num_features == 1:
    axes = [axes]

for idx, feature in enumerate(features):
    visualize_distribution(feature, axes[idx])

plt.show()


In [ ]:
def visualize_proportions(series):
    cleaned_series = series.dropna()

    categories = cleaned_series.value_counts().index.tolist()
    proportions = cleaned_series.value_counts().values.tolist()

    plt.figure(figsize=(8, 6))
    plt.bar(categories, proportions, color=['skyblue', 'orange'], edgecolor='black')
    plt.title(f"Category Proportions: {series.name.capitalize()}", fontsize=16)
    plt.xlabel(series.name.capitalize(), fontsize=12)
    plt.ylabel("Count", fontsize=12)
    plt.grid(visible=True, alpha=0.4)
    plt.tight_layout()
    plt.show()

visualize_proportions(df['sex'])


In [ ]:
def visualize_data_distribution(dataframe):
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), constrained_layout=True)

    sns.violinplot(x=dataframe['weight'].dropna(), ax=axes[0], color="lightcoral")
    axes[0].set_title("Weight Distribution", fontsize=14)
    axes[0].set_xlabel("Weight (kg)", fontsize=12)
    axes[0].set_ylabel("Density", fontsize=12)

    sns.violinplot(x=dataframe['height'].dropna(), ax=axes[1], color="lightgreen")
    axes[1].set_title("Height Distribution", fontsize=14)
    axes[1].set_xlabel("Height (m)", fontsize=12)
    axes[1].set_ylabel("Density", fontsize=12)

    sns.violinplot(x=dataframe['age'].dropna(), ax=axes[2], color="lightyellow")
    axes[2].set_title("Age Distribution", fontsize=14)
    axes[2].set_xlabel("Age (years)", fontsize=12)
    axes[2].set_ylabel("Density", fontsize=12)

    fig.suptitle("Distributions of Weight, Height, and Age", fontsize=16, y=1.05)

    plt.show()

visualize_data_distribution(df)

print(f"Mean Height: {df['height'].mean()} m")
print(f"Mean Weight: {df['weight'].mean()} kg")


# **IMAGE FRONT**

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Initialize counter for subplot positioning
x = 1

# Path to the image dataset
image_base_path = '/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/side/side/'

# Iterate through the filenames in the dataframe
for filename in df.id:
    if x < 10:  # Limit to the first 9 images
        # Construct the full path for the image
        img_name = f"{image_base_path}{filename}.jpg"

        # Read the image
        image = cv2.imread(img_name)

        if image is not None:
            print(image.shape)  # Print the shape of the image
            # Convert BGR (default in OpenCV) to RGB (used in Matplotlib)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Display the image in a 3x3 grid of subplots
            plt.subplot(3, 3, x)
            plt.axis("off")  # Turn off axis for better visualization
            plt.imshow(image_rgb)
            plt.title(f"ID: {filename}")  # Optionally add the ID as the title
            x += 1  # Increment the counter for the next subplot

# Adjust the layout to avoid overlap
plt.tight_layout()
plt.show()


# **IMAGE SIDE**

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Initialize counter for subplot positioning
x = 1

# Path to the image dataset
image_base_path = '/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/front/front/'

# Iterate through the filenames in the dataframe
for filename in df.id:
    if x < 10:  # Limit to the first 9 images
        # Construct the full path for the image
        img_name = f"{image_base_path}{filename}.jpg"

        # Read the image
        image = cv2.imread(img_name)

        if image is not None:
            print(image.shape)  # Print the shape of the image
            # Convert BGR (default in OpenCV) to RGB (used in Matplotlib)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Display the image in a 3x3 grid of subplots
            plt.subplot(3, 3, x)
            plt.axis("off")  # Turn off axis for better visualization
            plt.imshow(image_rgb)
            plt.title(f"ID: {filename}")  # Optionally add the ID as the title
            x += 1  # Increment the counter for the next subplot

# Adjust the layout to avoid overlap
plt.tight_layout()
plt.show()


In [ ]:
a = df.head()
print(a)
df.shape

In [ ]:
pip install tensorflow


# **SPLITING THE DATA**

In [ ]:
from tensorflow.keras.utils import to_categorical
from PIL import Image
import numpy as np

TRAIN_TEST_RATIO = 0.8

selected_columns = ['sex', 'race', 'eyes', 'hair']
unique_labels = {col: df[col].dropna().unique().tolist() for col in selected_columns}

race_categories = unique_labels['race']
eyes_categories = unique_labels['eyes']
hair_categories = unique_labels['hair']

category_mappings = {
    'race_labels': {race: idx for idx, race in enumerate(race_categories)},
    'eyes_labels': {eyes: idx for idx, eyes in enumerate(eyes_categories)},
    'hair_labels': {hair: idx for idx, hair in enumerate(hair_categories)}
}

class ImageDataLoader:
    def __init__(self, dataframe, img_width=224, img_height=224, dataset_dir='/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/'):
        self.dataframe = dataframe
        self.img_width = img_width
        self.img_height = img_height
        self.dataset_dir = dataset_dir

    def load_and_resize_image(self, image_path):
        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image = image.resize((self.img_width, self.img_height))
                image = np.array(image) / 255.0
            return image
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return None

    def create_train_test_split(self):
        if len(self.dataframe) == 0:
            raise ValueError("The DataFrame is empty. Cannot generate split indices.")
        if len(self.dataframe) < 3:
            raise ValueError("The DataFrame is too small for train/validation/test splits.")

        shuffled_indices = np.random.permutation(len(self.dataframe))
        train_split_end = int(len(self.dataframe) * TRAIN_TEST_RATIO)
        train_indices = shuffled_indices[:train_split_end]
        test_indices = shuffled_indices[train_split_end:]
        train_split_end = int(train_split_end * TRAIN_TEST_RATIO)
        train_indices, validation_indices = train_indices[:train_split_end], train_indices[train_split_end:]

        print(f"Training set size: {len(train_indices)}, Validation set size: {len(validation_indices)}, Test set size: {len(test_indices)}")

        self.dataframe['sex_encoded'] = self.dataframe['sex'].map({'Male': 0, 'Female': 1})
        if self.dataframe['sex_encoded'].isnull().any():
            raise ValueError("Some values in the 'sex' column do not match the expected values ('Male', 'Female').")

        self.max_bmi_value = self.dataframe['bmi'].max()

        return train_indices, validation_indices, test_indices

    def generate_batch(self, image_indices, is_training, batch_size=16):
        front_images_batch, side_images_batch, bmi_values, sex_labels = [], [], [], []
        while True:
            for idx in image_indices:
                individual = self.dataframe.iloc[idx]

                # Load front and side images
                front_image_path = f"{self.dataset_dir}/front/front/{individual['id']}.jpg"
                front_image = self.load_and_resize_image(front_image_path)

                side_image_path = f"{self.dataset_dir}/side/side/{individual['id']}.jpg"
                side_image = self.load_and_resize_image(side_image_path)

                if front_image is not None and side_image is not None:
                    front_images_batch.append(front_image)
                    side_images_batch.append(side_image)
                    bmi_values.append(individual['bmi'])
                    sex_labels.append(to_categorical(individual['sex_encoded'], 2))  # Binary encoding (0: male, 1: female)

                    # If batch is full, yield it and reset
                    if len(front_images_batch) >= batch_size:
                        # Yield images and their respective labels
                        yield [np.array(front_images_batch), np.array(side_images_batch)], [np.array(bmi_values), np.array(sex_labels)]
                        front_images_batch, side_images_batch, bmi_values, sex_labels = [], [], [], []

            # Break the loop if it's not training mode (validation/testing)
            if not is_training:
                break


image_loader = ImageDataLoader(df)

try:
    train_indices, validation_indices, test_indices = image_loader.create_train_test_split()
    print("Successfully generated training, validation, and test indices.")
except ValueError as e:
    print(f"Error: {e}")


In [ ]:
from tensorflow.keras.utils import to_categorical
from PIL import Image
import numpy as np
import random

DIVISION_RATIO = 0.8

attributes_to_map = ['sex', 'race', 'eyes', 'hair']
category_collections = {attr: df[attr].dropna().unique().tolist() for attr in attributes_to_map}

permitted_ethnicity = category_collections['race']
permitted_eye_colors = category_collections['eyes']
permitted_hair_colors = category_collections['hair']

category_mapping = {
    'permitted_ethnicity': {ethnicity: index for index, ethnicity in enumerate(permitted_ethnicity)},
    'permitted_eye_colors': {eye_color: index for index, eye_color in enumerate(permitted_eye_colors)},
    'permitted_hair_colors': {hair_color: index for index, hair_color in enumerate(permitted_hair_colors)}
}

class PersonImageProcessor:
    def __init__(self, dataframe, image_width=224, image_height=224, base_directory='/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/'):
        self.source_dataframe = dataframe
        self.IMAGE_WIDTH = image_width
        self.IMAGE_HEIGHT = image_height
        self.base_path = base_directory

    def transform_image(self, image_path):
        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image = image.resize((self.IMAGE_WIDTH, self.IMAGE_HEIGHT))
                image_array = np.array(image) / 255.0
            return image_array
        except Exception as processing_error:
            print(f"Image processing error for {image_path}: {processing_error}")
            return None

    def create_data_split_indices(self):
        if len(self.source_dataframe) == 0:
            raise ValueError("Input dataframe is empty. Cannot generate data splits.")
        if len(self.source_dataframe) < 3:
            raise ValueError("Dataframe is insufficient for train/validation/test splits.")

        randomized_indices = np.random.permutation(len(self.source_dataframe))
        split_point = int(len(self.source_dataframe) * DIVISION_RATIO)
        train_indices = randomized_indices[:split_point]
        test_indices = randomized_indices[split_point:]
        validation_split_point = int(split_point * DIVISION_RATIO)
        train_indices, validation_indices = train_indices[:validation_split_point], train_indices[validation_split_point:]

        print(f"Training set size: {len(train_indices)}, Validation set size: {len(validation_indices)}, Test set size: {len(test_indices)}")

        self.source_dataframe['gender_encoded'] = self.source_dataframe['sex'].map({'Male': 0, 'Female': 1})
        if self.source_dataframe['gender_encoded'].isnull().any():
            raise ValueError("Invalid gender values. Only 'Male' and 'Female' are accepted.")

        maximum_body_mass_index = self.source_dataframe['bmi'].max()

        return train_indices, validation_indices, test_indices

    def generate_image_batches(self, selected_indices, is_training_mode, batch_volume=16):
        frontal_images, profile_images, body_mass_indices, gender_labels = [], [], [], []
        while True:
            for index in selected_indices:
                individual_record = self.source_dataframe.iloc[index]

                frontal_image_path = f"/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/front/front/{individual_record['id']}.jpg"
                frontal_image = self.transform_image(frontal_image_path)

                profile_image_path = f"/root/.cache/kagglehub/datasets/davidjfisher/illinois-doc-labeled-faces-dataset/versions/1/side/side/{individual_record['id']}.jpg"
                profile_image = self.transform_image(profile_image_path)

                if frontal_image is not None and profile_image is not None:
                    frontal_images.append(frontal_image)
                    profile_images.append(profile_image)
                    body_mass_indices.append(individual_record['bmi'])
                    gender_labels.append(to_categorical(individual_record['gender_encoded'], 2))

                    if len(frontal_images) >= batch_volume:
                        yield [np.array(frontal_images), np.array(profile_images)], [np.array(body_mass_indices), np.array(gender_labels)]
                        frontal_images, profile_images, body_mass_indices, gender_labels = [], [], [], []

            if not is_training_mode:
                break

image_data_generator = PersonImageProcessor(df)

try:
    train_indices, validation_indices, test_indices = image_data_generator.create_data_split_indices()
    print("Data indices for training, validation, and testing generated successfully.")
except ValueError as error:
    print(f"Data generation error: {error}")

# **TRAINING MODEL**

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50

def coefficient_of_determination(y_actual, y_predicted):
    ss_total = K.sum(K.square(y_actual - K.mean(y_actual)))
    ss_residual = K.sum(K.square(y_actual - y_predicted))
    return 1 - ss_residual / (ss_total + K.epsilon())

def correlation_metric(y_actual, y_predicted):
    mean_actual = K.mean(y_actual)
    mean_predicted = K.mean(y_predicted)
    covariance = K.mean((y_actual - mean_actual) * (y_predicted - mean_predicted))
    std_actual = K.std(y_actual)
    std_predicted = K.std(y_predicted)
    return covariance / (std_actual * std_predicted + K.epsilon())

image_dimension = 224

# Load ResNet50 as the base model for both front and side images
resnet_base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(image_dimension, image_dimension, 3))

# Front Image Model
front_image_input = Input(shape=(image_dimension, image_dimension, 3), name="front_image_input")
front_feature_extraction = resnet_base_model(front_image_input)
front_features = GlobalAveragePooling2D()(front_feature_extraction)
front_features = Dropout(0.5)(front_features)
front_dense_features = Dense(128, activation='relu')(front_features)

# Side Image Model
side_image_input = Input(shape=(image_dimension, image_dimension, 3), name="side_image_input")
side_feature_extraction = resnet_base_model(side_image_input)
side_features = GlobalAveragePooling2D()(side_feature_extraction)
side_features = Dropout(0.5)(side_features)
side_dense_features = Dense(128, activation='relu')(side_features)

# Fusion Layer
combined_features = concatenate([front_dense_features, side_dense_features])
fusion_layer = Dense(256, activation="relu")(combined_features)
fusion_layer = Dropout(0.5)(fusion_layer)

# Outputs
body_mass_output = Dense(1, name="body_mass_output")(fusion_layer)  # Regression output
gender_output = Dense(2, activation='softmax', name="gender_output")(fusion_layer)  # Binary classification output

# Model Configuration
multitask_model = Model(inputs=[front_image_input, side_image_input], outputs=[body_mass_output, gender_output])

# Model Compilation
multitask_model.compile(
    optimizer=Adam(),
    loss={'body_mass_output': 'mse', 'gender_output': 'categorical_crossentropy'},
    metrics={
        'body_mass_output': ['mae', 'mse', coefficient_of_determination, correlation_metric],
        'gender_output': ['accuracy']
    }
)

multitask_model.summary()

# **TRAINING**

In [ ]:
import tensorflow as tf
import numpy as np

def generator_to_tensorflow_dataset(generator):
    def tf_generator():
        for [images, labels] in generator:
            # Unpack images into front and side inputs
            front_images, side_images = images

            # Unpack labels
            bmi_labels, gender_labels = labels

            yield (
                {
                    'front_image_input': front_images,
                    'side_image_input': side_images
                },
                {
                    'body_mass_output': bmi_labels,
                    'gender_output': gender_labels
                }
            )

    dataset = tf.data.Dataset.from_generator(
        tf_generator,
        output_signature=(
            {
                'front_image_input': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
                'side_image_input': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32)
            },
            {
                'body_mass_output': tf.TensorSpec(shape=(None,), dtype=tf.float32),
                'gender_output': tf.TensorSpec(shape=(None, 2), dtype=tf.float32)
            }
        )
    )

    return dataset.repeat()

# Training
training_history = multitask_model.fit(
    generator_to_tensorflow_dataset(
        image_data_generator.generate_image_batches(train_indices, is_training_mode=True, batch_volume=32)
    ),
    validation_data=generator_to_tensorflow_dataset(
        image_data_generator.generate_image_batches(validation_indices, is_training_mode=False, batch_volume=32)
    ),
    steps_per_epoch=len(train_indices) // 32,
    validation_steps=len(validation_indices) // 32,
    epochs=10
)

In [ ]:
multitask_model.save("multitask_model.h5")

In [ ]:
multitask_model.save("multitask_model.keras")

***LOADING THE MODEL***

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
file_path = '/content/drive/MyDrive/Prml project model h5/multitask_model.h5'


In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K

def coefficient_of_determination(y_actual, y_predicted):
    ss_total = K.sum(K.square(y_actual - K.mean(y_actual)))
    ss_residual = K.sum(K.square(y_actual - y_predicted))
    return 1 - ss_residual / (ss_total + K.epsilon())

def correlation_metric(y_actual, y_predicted):
    mean_actual = K.mean(y_actual)
    mean_predicted = K.mean(y_predicted)
    covariance = K.mean((y_actual - mean_actual) * (y_predicted - mean_predicted))
    std_actual = K.std(y_actual)
    std_predicted = K.std(y_predicted)
    return covariance / (std_actual * std_predicted + K.epsilon())

# Recreate the model architecture exactly as before
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, concatenate
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

image_dimension = 224
resnet_base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(image_dimension, image_dimension, 3))

# Front Image Model
front_image_input = Input(shape=(image_dimension, image_dimension, 3), name="front_image_input")
front_feature_extraction = resnet_base_model(front_image_input)
front_features = GlobalAveragePooling2D()(front_feature_extraction)
front_features = Dropout(0.5)(front_features)
front_dense_features = Dense(128, activation='relu')(front_features)

# Side Image Model
side_image_input = Input(shape=(image_dimension, image_dimension, 3), name="side_image_input")
side_feature_extraction = resnet_base_model(side_image_input)
side_features = GlobalAveragePooling2D()(side_feature_extraction)
side_features = Dropout(0.5)(side_features)
side_dense_features = Dense(128, activation='relu')(side_features)

# Fusion Layer
combined_features = concatenate([front_dense_features, side_dense_features])
fusion_layer = Dense(256, activation="relu")(combined_features)
fusion_layer = Dropout(0.5)(fusion_layer)

# Outputs
body_mass_output = Dense(1, name="body_mass_output")(fusion_layer)
gender_output = Dense(2, activation='softmax', name="gender_output")(fusion_layer)

# Recreate the Model
loaded_model = Model(inputs=[front_image_input, side_image_input], outputs=[body_mass_output, gender_output])

# Compile the model with the same configuration
loaded_model.compile(
    optimizer=Adam(),
    loss={'body_mass_output': 'mse', 'gender_output': 'categorical_crossentropy'},
    metrics={
        'body_mass_output': ['mae', 'mse', coefficient_of_determination, correlation_metric],
        'gender_output': ['accuracy']
    }
)

# Load the weights
file_path = '/content/multitask_model.h5'
loaded_model.load_weights(file_path)

# Verify the loaded model
loaded_model.summary()

**Graph** **Measurements**

In [ ]:
import matplotlib.pyplot as plt

def visualize_model_performance(training_history):
    # BMI Metrics
    bmi_metrics = {
        'mae': training_history.history['body_mass_output_mae'],
        'mse': training_history.history['body_mass_output_mse'],
        'r2': training_history.history['body_mass_output_coefficient_of_determination'],
        'pearson': training_history.history['body_mass_output_correlation_metric']
    }

    # Gender Metrics
    gender_metrics = training_history.history['gender_output_accuracy']

    # BMI Metrics Visualization
    plt.figure(figsize=(15, 10))

    metric_details = [
        ('MAE', bmi_metrics['mae'], 1),
        ('MSE', bmi_metrics['mse'], 2),
        ('R²', bmi_metrics['r2'], 3),
        ('Pearson', bmi_metrics['pearson'], 4)
    ]

    for title, values, subplot_pos in metric_details:
        plt.subplot(2, 2, subplot_pos)
        plt.plot(values, label=f'Training {title}')
        plt.title(f'Body Mass Index {title}')
        plt.xlabel('Epochs')
        plt.ylabel(title)
        plt.legend()

    # Gender Metrics Visualization
    plt.figure(figsize=(8, 6))
    plt.plot(gender_metrics, label='Training Accuracy')
    plt.title('Gender Classification Performance')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Call the function with the training history from the model training
visualize_model_performance(training_history)

**TESTING**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

test_batch_size = 16

test_generator = image_loader.generate_batch(test_indices, is_training=False, batch_size=test_batch_size)


bmi_true = []
sex_true = []

bmi_predictions = []
sex_predictions = []

for images, labels in test_generator:
    bmi_true.append(labels[0])
    sex_true.append(labels[1])

    preds = multitask_model.predict([images[0], images[1]], batch_size=test_batch_size)

    bmi_predictions.append(preds[0])
    sex_predictions.append(preds[1])

bmi_true = np.concatenate(bmi_true)
sex_true = np.concatenate(sex_true)

bmi_predictions = np.concatenate(bmi_predictions)
sex_predictions = np.concatenate(sex_predictions)

In [ ]:
len(bmi_predictions)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.scatter(bmi_true, bmi_predictions, alpha=0.7, label="Predictions", color="blue")
plt.plot([15, 40], [15, 40], color="red", linestyle="--", label="Ideal")
plt.title("True BMI vs Predicted BMI")
plt.xlabel("True BMI")
plt.ylabel("Predicted BMI")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.scatter(bmi_true, bmi_predictions, alpha=0.7, label="Predictions", color="blue")
plt.plot([0, max(bmi_true)], [0, max(bmi_predictions)], color="red", linestyle="--", label="Ideal")
plt.title("True BMI vs Predicted BMI")
plt.xlabel("True BMI")
plt.ylabel("Predicted BMI")

# Adjust axis limits to focus on the data range
plt.xlim(0, max(bmi_true) * 1.1)
plt.ylim(0, max(bmi_predictions) * 1.1)

plt.legend()
plt.grid(True)
plt.show()


# **METRICS**

In [ ]:
bmi_mae = mean_absolute_error(bmi_true, bmi_predictions)
bmi_mse = mean_squared_error(bmi_true, bmi_predictions)
bmi_r2 = r2_score(bmi_true, bmi_predictions)
bmi_pearson, _ = pearsonr(bmi_true.flatten(), bmi_predictions.flatten())  # Flatten ensures arrays are 1D

print(f"BMI MAE: {bmi_mae}")
print(f"BMI MSE: {bmi_mse}")
print(f"BMI R²: {bmi_r2}")
print(f"BMI Pearson Correlation Coefficient: {bmi_pearson}")


In [ ]:
from sklearn.metrics import accuracy_score

sex_accuracy = accuracy_score(sex_true, np.round(sex_predictions))

print(f"Sex Accuracy: {sex_accuracy}")

In [ ]:
img_width = 224
img_height = 224

def preprocess_image(img_path):
    try:
        img = Image.open(img_path)
        img = img.convert("RGB")
        img = img.resize((img_width, img_height))
        img_array = np.array(img) / 255.0
        return img_array
    except Exception as e:
        print(f"Error processing image {img_path}: {e}")
        return None

def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 24.9:
        return 'Normal'
    else:
        return 'Overweight'

def customImageTest(front_image_path, side_image_path):
    front_image = preprocess_image(front_image_path)
    side_image = preprocess_image(side_image_path)

    if front_image is None or side_image is None:
        print("One of the images is invalid. Please check the file paths.")
    else:
        front_image = np.expand_dims(front_image, axis=0)  # Add batch dimension
        side_image = np.expand_dims(side_image, axis=0)  # Add batch dimension

        bmi_prediction, sex_prediction = multitask_model.predict([front_image, side_image])

        predicted_bmi = bmi_prediction[0][0]
        predicted_sex = np.argmax(sex_prediction[0])

        bmi_category = categorize_bmi(predicted_bmi)

        print(f"Predicted BMI: {predicted_bmi}")
        print(f"BMI Category: {bmi_category}")
        print(f"Predicted Sex: {'Male' if predicted_sex == 0 else 'Female'}")

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import os
print(os.getcwd())


In [ ]:
os.chdir('/content')


# **TESTING WITH FRIEND IMAGES**

In [ ]:
import os

# List files in the current working directory
print(os.listdir('/content'))


In [ ]:
print(os.path.exists('/content/chand_front.jpg'))  # Check for front.jpg
print(os.path.exists('/content/chand_side.jpg'))   # Check for side.jpg


In [ ]:
customImageTest('/content/chand_front.jpg', '/content/chand_side.jpg')
